In [0]:
base_path = "/Volumes/ev-charge/default/ev-data"

stations_path = "/Volumes/ev-charge/default/ev-data/stations.csv"
maintenance_path = "/Volumes/ev-charge/default/ev-data/maintenance.csv"
chargers_path = "/Volumes/ev-charge/default/ev-data/chargers.json"
sessions_path = "/Volumes/ev-charge/default/ev-data/sessions.parquet"

In [0]:
df_stations = spark.read.option("header", "true").option("inferSchema", "true").csv(stations_path)

df_maintenance = spark.read.option("header", "true").option("inferSchema", "true").csv(maintenance_path)

df_chargers = spark.read.option("multiline", "true").json(chargers_path)


In [0]:
import pandas as pd

pdf = pd.read_parquet(sessions_path)

In [0]:
for col in pdf.columns:
    if str(pdf[col].dtype).startswith("datetime"):
        pdf[col] = pdf[col].astype("datetime64[ms]")

In [0]:
df_sessions = spark.createDataFrame(pdf)

In [0]:
display(dbutils.fs.ls("/Volumes/ev-charge/default/ev-data/"))

path,name,size,modificationTime
dbfs:/Volumes/ev-charge/default/ev-data/chargers.json,chargers.json,436543,1785490823000
dbfs:/Volumes/ev-charge/default/ev-data/maintenance.csv,maintenance.csv,3471904,1785490824000
dbfs:/Volumes/ev-charge/default/ev-data/sessions.parquet,sessions.parquet,15772650,1785490827000
dbfs:/Volumes/ev-charge/default/ev-data/stations.csv,stations.csv,28931,1785490823000


In [0]:
df_stations.createOrReplaceTempView("stations_raw")
df_maintenance.createOrReplaceTempView("maintenance_raw")
df_chargers.createOrReplaceTempView("chargers_raw")
df_sessions.createOrReplaceTempView("sessions_raw")

In [0]:
stations_count = df_stations.count()
maintenance_count = df_maintenance.count()
chargers_count = df_chargers.count()
sessions_count = df_sessions.count()

print(stations_count, maintenance_count, chargers_count, sessions_count)

180 18000 1200 300000


In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

stations_bronze = df_stations \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("run_id", lit("run_001"))

stations_bronze.createOrReplaceTempView("stations_bronze_view")

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

maintenance_bronze = df_maintenance.withColumn("ingestion_time", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("run_id", lit("run_001"))

maintenance_bronze.createOrReplaceTempView("maintenance_bronze_view")

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

chargers_bronze = df_chargers \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .withColumn("run_id", lit("run_001"))

chargers_bronze.createOrReplaceTempView("chargers_bronze_view")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

sessions_bronze = df_sessions \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("source_file", lit("sessions_parquet")) \
    .withColumn("run_id", lit("run_001"))

sessions_bronze.createOrReplaceTempView("sessions_bronze_view")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze_layer")

DataFrame[]

In [0]:
stations_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze_layer.bronze_stations")

maintenance_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze_layer.bronze_maintenance")

chargers_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze_layer.bronze_chargers")

sessions_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze_layer.bronze_sessions")

In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()

+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|        workspace|         default|
+-----------------+----------------+



In [0]:
spark.sql("SHOW TABLES IN workspace.default").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|     bronze_chargers|      false|
| default|  bronze_maintenance|      false|
| default|     bronze_sessions|      false|
| default|     bronze_stations|      false|
| default|shiptrack_week03_...|      false|
| default|shiptrack_week03_...|      false|
|        |chargers_bronze_view|       true|
|        |        chargers_raw|       true|
|        |maintenance_bronz...|       true|
|        |     maintenance_raw|       true|
|        |sessions_bronze_view|       true|
|        |        sessions_raw|       true|
|        |stations_bronze_view|       true|
|        |        stations_raw|       true|
+--------+--------------------+-----------+



In [0]:
stations_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_stations")

maintenance_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_maintenance")

chargers_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_chargers")

sessions_bronze.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_sessions")

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|     bronze_chargers|      false|
| default|  bronze_maintenance|      false|
| default|     bronze_sessions|      false|
| default|     bronze_stations|      false|
| default|shiptrack_week03_...|      false|
| default|shiptrack_week03_...|      false|
|        |chargers_bronze_view|       true|
|        |        chargers_raw|       true|
|        |maintenance_bronz...|       true|
|        |     maintenance_raw|       true|
|        |sessions_bronze_view|       true|
|        |        sessions_raw|       true|
|        |stations_bronze_view|       true|
|        |        stations_raw|       true|
+--------+--------------------+-----------+



In [0]:
bronze_stations_count = spark.table("workspace.default.bronze_stations").count()
bronze_maintenance_count = spark.table("workspace.default.bronze_maintenance").count()
bronze_chargers_count = spark.table("workspace.default.bronze_chargers").count()
bronze_sessions_count = spark.table("workspace.default.bronze_sessions").count()

print("Stations:", stations_count, bronze_stations_count)
print("Maintenance:", maintenance_count, bronze_maintenance_count)
print("Chargers:", chargers_count, bronze_chargers_count)
print("Sessions:", sessions_count, bronze_sessions_count)

Stations: 180 180
Maintenance: 18000 18000
Chargers: 1200 1200
Sessions: 300000 300000


In [0]:
def check_match(src, tgt, name):
    if src == tgt:
        print(f"✅ {name} MATCH")
    else:
        print(f"❌ {name} MISMATCH")

check_match(stations_count, bronze_stations_count, "Stations")
check_match(maintenance_count, bronze_maintenance_count, "Maintenance")
check_match(chargers_count, bronze_chargers_count, "Chargers")
check_match(sessions_count, bronze_sessions_count, "Sessions")

✅ Stations MATCH
✅ Maintenance MATCH
✅ Chargers MATCH
✅ Sessions MATCH
